# 3. Dimensionality Reduction

## Part A: Steady state



In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from dimred_utils import load_comsol_export

In [ ]:
coords_steady, t_steady, X_steady = load_comsol_export("dimred_steady.txt")

print(f"Steady Matrix    : {X_steady.shape} (Points x Time Steps)")
print(f"Steady Times     : {t_steady[0]:.4f} s to {t_steady[-1]:.4f} s")

### Principal Component Analysis (PCA)

In [ ]:
# --- 1. Fit PCA with All Available Components ---
# Input shape: (n_samples, n_features) -> (N_timesteps, N_points)
pca_full = PCA()
pca_full.fit(X_steady.T)

var_ratio = pca_full.explained_variance_ratio_ * 100
cum_var_ratio = np.cumsum(var_ratio)
n_components = np.arange(1, len(var_ratio) + 1)

# --- 2. Plot Explained Variance ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(n_components[:15], var_ratio[:15], 'o-', color='tab:blue', linewidth=1.5, markersize=6)
plt.title('Individual Explained Variance')
plt.xlabel('Component Number')
plt.ylabel('Variance Ratio / %')
plt.grid(True, linestyle=':', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(n_components[:15], cum_var_ratio[:15], 's-', color='tab:red', linewidth=1.5, markersize=6)
plt.axhline(99.0, color='gray', linestyle='--', alpha=0.7, label='99% Threshold')
plt.title('Cumulative Explained Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance / %')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# --- Perform PCA ---
N_components = 4 # adjust as needed based on the previous results
pca = PCA(n_components=N_components)
pca.fit(X_steady.T)

# Spatial components (loadings) of shape (n_components, N_points)
modes = pca.components_
variance_ratio = pca.explained_variance_ratio_

plt.figure(figsize=(15, 8))

for i in range(N_components):
    mode_spatial = modes[i, :]
    c_lim = np.max(np.abs(mode_spatial))
    
    plt.subplot(2, 3, i + 1)
    plt.scatter(
        coords_steady[:, 0], coords_steady[:, 1],
        c=mode_spatial, cmap='RdBu_r',
        vmin=-c_lim, vmax=c_lim,
        s=10, edgecolors='none'
    )
    plt.colorbar(label=r'$\phi_{' + str(i+1) + r'}\,/\,\mathrm{a.u.}$')
    plt.axis('equal')
    plt.title(rf'$\mathrm{{Mode}}\ {i+1}\ ({variance_ratio[i]*100:.1f}\%)$')
    plt.xlabel(r'$x\,/\,\mathrm{m}$')
    plt.ylabel(r'$y\,/\,\mathrm{m}$')

plt.tight_layout()
plt.show()


Reconstruct a snapshot with PCA and plot comparison to the original snapshot:

In [ ]:
sample_idx = 15  # Time step index to reconstruct (0 to N_timesteps - 1)

# Project snapshots into PCA space and inverse transform back to spatial domain
X_proj = pca.transform(X_steady.T)
X_recon = pca.inverse_transform(X_proj).T

# Extract fields for the selected time step
orig_sample = X_steady[:, sample_idx]
pred_sample = X_recon[:, sample_idx]

# Relative error normalized by peak field amplitude to avoid division-by-zero at nodal lines
p_max = np.max(np.abs(orig_sample))
rel_diff_pct = ((orig_sample - pred_sample) / p_max) * 100

plt.figure(figsize=(15, 4.5))
# 1. Original Snapshot
plt.subplot(1, 3, 1)
plt.scatter(
    coords_steady[:, 0], coords_steady[:, 1],
    c=orig_sample, cmap='RdBu_r',
    vmin=-p_max, vmax=p_max,
    s=10, edgecolors='none'
)
plt.colorbar(label='p / Pa')
plt.axis('equal')
plt.title(f'Original (t = {t_steady[sample_idx]*1000:.2f} ms)')
plt.xlabel('x / m')
plt.ylabel('y / m')

# 2. PCA Reconstruction
plt.subplot(1, 3, 2)
plt.scatter(
    coords_steady[:, 0], coords_steady[:, 1],
    c=pred_sample, cmap='RdBu_r',
    vmin=-p_max, vmax=p_max,
    s=10, edgecolors='none'
)
plt.colorbar(label='p / Pa')
plt.axis('equal')
plt.title(f'PCA Reconstruction (N = {N_components})')
plt.xlabel('x / m')
plt.ylabel('y / m')

# 3. Normalized Difference (%)
plt.subplot(1, 3, 3)
err_lim = np.max(np.abs(rel_diff_pct))
plt.scatter(
    coords_steady[:, 0], coords_steady[:, 1],
    c=rel_diff_pct, cmap='PiYG_r',
    vmin=-err_lim, vmax=err_lim,
    s=10, edgecolors='none'
)
plt.colorbar(label='Error / %')
plt.axis('equal')
plt.title('Difference / %')
plt.xlabel('x / m')
plt.ylabel('y / m')

plt.tight_layout()
plt.show()

### Dynamic Mode Decomposition (DMD)

Here we use DMD function from Brunton & Kutz, but for more real use, there is a Python package PyDMD.

In [ ]:
# --- DMD Implementation (Brunton & Kutz) ---
def DMD(X, Xprime, r):
    # Step 1: Truncated SVD
    U, Sigma, VT = np.linalg.svd(X, full_matrices=0)
    Ur = U[:, :r]
    Sigmar = np.diag(Sigma[:r])
    VTr = VT[:r, :]
    
    # Step 2: Low-rank linear operator
    Atilde = np.linalg.solve(Sigmar.T, (Ur.T @ Xprime @ VTr.T).T).T
    
    # Step 3: Spectral decomposition of Atilde
    Lambda, W = np.linalg.eig(Atilde)
    Lambda = np.diag(Lambda)
    
    # Step 4: Exact DMD modes and mode amplitudes
    Phi = Xprime @ np.linalg.solve(Sigmar.T, VTr).T @ W
    alpha1 = Sigmar @ VTr[:, 0]
    b = np.linalg.solve(W @ Lambda, alpha1)
    
    return Phi, Lambda, b

In [ ]:
# --- Prepare Snapshot Pairs ---
X1 = X_steady[:, :-1]
X2 = X_steady[:, 1:]

# --- Economy SVD for Rank Diagnostics ---
_, S_vec, _ = np.linalg.svd(X1, full_matrices=False)
variance_pct = ((S_vec**2) / np.sum(S_vec**2)) * 100
cumulative_energy = np.cumsum(variance_pct)

plt.figure(figsize=(11, 4.5))

# Subplot 1: Variance (%) per Mode
plt.subplot(1, 2, 1)
plt.semilogy(range(1, len(S_vec) + 1), variance_pct, '+-', color='tab:blue', markersize=4)
plt.title('Singular Value Spectrum (POD)')
plt.xlabel('Mode index')
plt.ylabel('Variance (%)')
plt.grid(True, linestyle=':', alpha=0.6)

# Subplot 2: Cumulative Energy (%)
plt.subplot(1, 2, 2)
plt.plot(range(1, len(S_vec) + 1), cumulative_energy, '+-', color='tab:green', markersize=4)
plt.axhline(99.0, color='gray', linestyle=':', label='99% Threshold')
plt.axhline(99.9, color='k', linestyle=':', label='99.9% Threshold')
plt.title('Cumulative Energy')
plt.xlabel('Mode index')
plt.ylabel('Energy (%)')
plt.ylim(min(cumulative_energy[0]-1, 80), 101)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

# Print quantitative breakdown for the first 10 ranks
print("--- Cumulative Energy by Rank ---")
for k in range(min(10, len(cumulative_energy))):
    print(f"Rank r = {k+1:2d} -> Cumulative Energy: {cumulative_energy[k]:.4f}% | Individual Variance: {variance_pct[k]:.4f}%")

In [ ]:
# Set the chosen rank based on the cumulative energy
r = 6

# Compute DMD
Phi, Lambda, b = DMD(X1, X2, r)
eigs_vals = np.diag(Lambda)

# Plot DMD Spectrum on the Unit Circle
plt.figure(figsize=(5, 5))
theta = np.linspace(0, 2 * np.pi, 101)
plt.plot(np.cos(theta), np.sin(theta), 'k--', label='Unit circle')
plt.scatter(np.real(eigs_vals), np.imag(eigs_vals), c='red', s=50, zorder=3, label='Ritz values')

plt.gca().set_aspect('equal', adjustable='box')
plt.xlim([-1.15, 1.15])
plt.ylim([-1.15, 1.15])
plt.grid(True, linestyle=':', alpha=0.6)
plt.title(f'DMD Spectrum (r = {r})')
plt.xlabel(r'$\mathrm{Re}(\lambda)$')
plt.ylabel(r'$\mathrm{Im}(\lambda)$')
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# --- Compute Continuous Frequencies ---
dt = np.mean(np.diff(t_steady))
omega = np.log(np.diag(Lambda)) / dt
freqs = np.imag(omega) / (2.0 * np.pi)

# --- Filter Unique Physical Modes (Drop Duplicate Conjugate Pairs) ---
unique_mode_indices = []
visited = set()

print("--- DMD Mode Analysis: Conjugate Pairs & Frequencies ---")
for i in range(len(Lambda)):
    if i in visited:
        continue
    
    val_i = Lambda[i, i]
    
    # Static or pure decay mode
    if np.isclose(np.imag(val_i), 0.0, atol=1e-6):
        unique_mode_indices.append(i)
        visited.add(i)
        print(f"Mode {i+1}: Real (Static/Decay)   | Freq: {freqs[i]:6.2f} Hz | λ: {val_i:.4f}")
    else:
        # Complex conjugate pair
        found_pair = False
        for j in range(i + 1, len(Lambda)):
            if j not in visited and np.isclose(val_i, np.conj(Lambda[j, j]), atol=1e-5):
                unique_mode_indices.append(i)
                visited.add(i)
                visited.add(j)
                print(f"Modes {i+1} & {j+1}: Conjugate Pair  | Freq: ±{abs(freqs[i]):5.2f} Hz | λ: {val_i:.4f}")
                found_pair = True
                break
        
        if not found_pair:
            unique_mode_indices.append(i)
            visited.add(i)
            print(f"Mode {i+1}: Unpaired Complex    | Freq: {freqs[i]:6.2f} Hz | λ: {val_i:.4f}")

# --- Plot Spatial Fields for Unique Modes Only ---
x, y = coords_steady[:, 0], coords_steady[:, 1]
n_unique = len(unique_mode_indices)

fig, axes = plt.subplots(n_unique, 2, figsize=(12, 2.6 * n_unique), squeeze=False)

for row, idx in enumerate(unique_mode_indices):
    mode_real = np.real(Phi[:, idx])
    mode_imag = np.imag(Phi[:, idx])
    max_val = max(np.max(np.abs(mode_real)), np.max(np.abs(mode_imag)))

    # Real Part
    sc0 = axes[row, 0].scatter(
        x, y, c=mode_real, cmap="RdBu_r", s=10,
        vmin=-max_val, vmax=max_val, edgecolors="none"
    )
    axes[row, 0].set_title(f"Mode {idx+1} (Real) - {abs(freqs[idx]):.2f} Hz")
    axes[row, 0].set_xlabel("x / m")
    axes[row, 0].set_ylabel("y / m")
    axes[row, 0].axis("equal")
    plt.colorbar(sc0, ax=axes[row, 0], fraction=0.03, pad=0.04)

    # Imaginary Part
    sc1 = axes[row, 1].scatter(
        x, y, c=mode_imag, cmap="RdBu_r", s=10,
        vmin=-max_val, vmax=max_val, edgecolors="none"
    )
    axes[row, 1].set_title(f"Mode {idx+1} (Imaginary) - {abs(freqs[idx]):.2f} Hz")
    axes[row, 1].set_xlabel("x / m")
    axes[row, 1].set_ylabel("y / m")
    axes[row, 1].axis("equal")
    plt.colorbar(sc1, ax=axes[row, 1], fraction=0.03, pad=0.04)

plt.tight_layout()
plt.show()

Reconstruct a snapshot with DMD and plot comparison to the original snapshot:

In [ ]:
# --- Pick a Single Time Step / Sample Index to Reconstruct ---
k_sample = 25 # choose anything from the available range, play around
t_sample = t_steady[k_sample]
orig_sample = X_steady[:, k_sample]

# --- DMD Single-Sample Reconstruction ---
evals_diag = np.diag(Lambda) if Lambda.ndim == 2 else Lambda
pred_sample = np.real(Phi @ ((evals_diag ** k_sample) * b))

# Relative Difference (%) Normalized by Peak Absolute Amplitude
p_max = np.max(np.abs(orig_sample))
rel_diff_pct = ((orig_sample - pred_sample) / p_max) * 100

rel_l2_error = np.linalg.norm(orig_sample - pred_sample) / np.linalg.norm(orig_sample)

print(f"Sample index          : {k_sample}")
print(f"Time                  : {t_sample:.4f} s")
print(f"Relative L2 Error     : {rel_l2_error * 100:.3f}%")
print(f"Max Relative Diff (%) : {np.max(np.abs(rel_diff_pct)):.3f}%")

# --- 3. Visualization ---
x_coords = coords_steady[:, 0]
y_coords = coords_steady[:, 1]
vmin, vmax = np.min(orig_sample), np.max(orig_sample)

# Symmetric bounds for divergence colormap on error
max_err_bound = max(abs(np.min(rel_diff_pct)), abs(np.max(rel_diff_pct)))

plt.figure(figsize=(16, 4))

# Subplot 1: Ground Truth
plt.subplot(1, 3, 1)
sc0 = plt.scatter(x_coords, y_coords, c=orig_sample, cmap="RdBu_r", s=12, vmin=vmin, vmax=vmax)
plt.title(f"Ground Truth")
plt.xlabel("x / m")
plt.ylabel("y / m")
plt.axis("equal")
plt.colorbar(sc0, fraction=0.046, pad=0.04)

# Subplot 2: DMD Reconstruction
plt.subplot(1, 3, 2)
sc1 = plt.scatter(x_coords, y_coords, c=pred_sample, cmap="RdBu_r", s=12, vmin=vmin, vmax=vmax)
plt.title(f"DMD Reconstruction (r = {r})")
plt.xlabel("x / m")
plt.ylabel("y / m")
plt.axis("equal")
plt.colorbar(sc1, fraction=0.046, pad=0.04)

# Subplot 3: Relative Difference (%)
plt.subplot(1, 3, 3)
sc2 = plt.scatter(
    x_coords, y_coords, c=rel_diff_pct, cmap="PiYG_r", s=12,
    vmin=-max_err_bound, vmax=max_err_bound
)
plt.title("Rel. Diff. (%)")
plt.xlabel("x / m")
plt.ylabel("y / m")
plt.axis("equal")
cb2 = plt.colorbar(sc2, fraction=0.046, pad=0.04)
cb2.set_label("% error")

plt.tight_layout()
plt.show()